## STATUS: KEPT

Data loading and cleaning pipeline. Produces the processed dataset used across all models.

In [2]:
# ============================================
# DATASET EXPLORATION FOR MBTI CLASSIFIER & AUTOENCODER
# ============================================

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

# ============================================
# 1. EXPLORE AUTOENCODER DATASETS
# ============================================

print("="*80)
print("AUTOENCODER DATASETS EXPLORATION")
print("="*80)

# Define paths
AUTOENCODER_1M_PATH = '/content/drive/MyDrive/mbti-tune/data/raw/spotify_data.csv'
AUTOENCODER_110K_PATH = '/content/drive/MyDrive/mbti-tune/data/raw/spotify_tracks.csv'

# Check if files exist
print(f"1M dataset exists: {os.path.exists(AUTOENCODER_1M_PATH)}")
print(f"110K dataset exists: {os.path.exists(AUTOENCODER_110K_PATH)}")
print()

# Function to explore dataset
def explore_autoencoder_dataset(filepath, name):
    print(f"\n{'='*50}")
    print(f"EXPLORING: {name}")
    print(f"{'='*50}")

    # Read only first few rows to check structure (to avoid memory issues)
    try:
        # Check file size
        file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
        print(f"File size: {file_size:.2f} MB")

        # Read first 5 rows to check columns
        sample = pd.read_csv(filepath, nrows=5)
        print(f"\nShape (sample): {sample.shape}")
        print(f"\nColumns ({len(sample.columns)} columns):")
        print(sample.columns.tolist())

        print(f"\nFirst 2 rows:")
        print(sample.head(2))

        print(f"\nData types:")
        print(sample.dtypes)

        # Read more to check for nulls and statistics
        # For large files, read in chunks
        if file_size > 100:  # For 1M dataset
            print("\nReading in chunks for statistics...")
            chunk_size = 10000
            total_rows = 0
            null_counts = pd.Series(dtype='float64')
            numeric_stats = {}

            for chunk in pd.read_csv(filepath, chunksize=chunk_size):
                total_rows += len(chunk)
                null_counts = null_counts.add(chunk.isnull().sum(), fill_value=0)

                # Collect numeric stats
                for col in chunk.select_dtypes(include=[np.number]).columns:
                    if col not in numeric_stats:
                        numeric_stats[col] = {'min': [], 'max': [], 'mean': [], 'std': []}
                    numeric_stats[col]['min'].append(chunk[col].min())
                    numeric_stats[col]['max'].append(chunk[col].max())
                    numeric_stats[col]['mean'].append(chunk[col].mean())
                    numeric_stats[col]['std'].append(chunk[col].std())

                if total_rows >= 100000:  # Sample enough rows
                    break

            print(f"\nTotal rows sampled: {total_rows}")
            print(f"\nNull counts (first 10 columns):")
            print(null_counts.head(10))

            print(f"\nNumeric columns statistics (first 5 columns):")
            for col in list(numeric_stats.keys())[:5]:
                print(f"\n{col}:")
                print(f"  Min: {min(numeric_stats[col]['min']):.4f}")
                print(f"  Max: {max(numeric_stats[col]['max']):.4f}")
                print(f"  Mean: {np.mean(numeric_stats[col]['mean']):.4f}")
                print(f"  Std: {np.mean(numeric_stats[col]['std']):.4f}")

        else:  # For 110K dataset
            df = pd.read_csv(filepath)
            print(f"\nFull shape: {df.shape}")
            print(f"\nNull counts (first 10 columns):")
            print(df.isnull().sum().head(10))

            print(f"\nNumeric columns statistics (first 5 columns):")
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            for col in numeric_cols[:5]:
                print(f"\n{col}:")
                print(f"  Min: {df[col].min():.4f}")
                print(f"  Max: {df[col].max():.4f}")
                print(f"  Mean: {df[col].mean():.4f}")
                print(f"  Std: {df[col].std():.4f}")
                print(f"  Nulls: {df[col].isnull().sum()}")

            print(f"\nSample of categorical columns:")
            categorical_cols = df.select_dtypes(include=['object']).columns
            for col in categorical_cols[:3]:
                print(f"\n{col} unique values (first 5):")
                print(df[col].value_counts().head(5))

        return sample

    except Exception as e:
        print(f"Error reading {name}: {e}")
        return None

# Explore both datasets
sample_1m = explore_autoencoder_dataset(AUTOENCODER_1M_PATH, "1M Songs Dataset")
sample_110k = explore_autoencoder_dataset(AUTOENCODER_110K_PATH, "110K Songs Dataset")

# ============================================
# 2. EXPLORE PLAYLIST DATASETS FOR CLASSIFIER
# ============================================

print("\n")
print("="*80)
print("CLASSIFIER DATASETS EXPLORATION")
print("="*80)

# Path for raw playlists
RAW_PLAYLIST_PATH = '/content/drive/MyDrive/mbti-tune/data/raw/raw_playlists/'

# Check if directory exists
print(f"Raw playlists directory exists: {os.path.exists(RAW_PLAYLIST_PATH)}")
if os.path.exists(RAW_PLAYLIST_PATH):
    # List personality folders
    personality_folders = [f for f in os.listdir(RAW_PLAYLIST_PATH)
                          if os.path.isdir(os.path.join(RAW_PLAYLIST_PATH, f))]
    print(f"\nPersonality types found: {personality_folders}")

    # Check one folder and one CSV file
    if personality_folders:
        first_personality = personality_folders[0]
        folder_path = os.path.join(RAW_PLAYLIST_PATH, first_personality)
        csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

        print(f"\nExploring: {first_personality}")
        print(f"Number of CSV files in {first_personality}: {len(csv_files)}")

        if csv_files:
            first_csv = csv_files[0]
            csv_path = os.path.join(folder_path, first_csv)
            print(f"\nSample CSV file: {first_csv}")

            # Read the CSV file
            df_playlist = pd.read_csv(csv_path)
            print(f"\nShape of playlist file: {df_playlist.shape}")
            print(f"\nColumns in playlist file:")
            print(df_playlist.columns.tolist())

            print(f"\nFirst 3 rows of playlist file:")
            print(df_playlist.head(3))

            print(f"\nData types:")
            print(df_playlist.dtypes)

            print(f"\nBasic statistics:")
            print(df_playlist.describe())

            print(f"\nNull values:")
            print(df_playlist.isnull().sum())

            # Check if there's a track ID or similar identifier
            print(f"\nSample of first column (possible ID):")
            print(df_playlist.iloc[:, 0].head(10))

            # Show which columns are numeric for autoencoder
            numeric_cols = df_playlist.select_dtypes(include=[np.number]).columns
            print(f"\nNumeric columns ({len(numeric_cols)}):")
            print(numeric_cols.tolist())

# ============================================
# 3. EXPLORE MBTI PLAYLIST MEANS DATASET
# ============================================

print("\n")
print("="*80)
print("MBTI PLAYLIST MEANS DATASET EXPLORATION")
print("="*80)

MBTI_PLAYLIST_PATH = '/content/drive/MyDrive/mbti-tune/data/raw/mbti_playlists/'

print(f"MBTI playlists directory exists: {os.path.exists(MBTI_PLAYLIST_PATH)}")
if os.path.exists(MBTI_PLAYLIST_PATH):
    # List all CSV files
    mbti_files = [f for f in os.listdir(MBTI_PLAYLIST_PATH) if f.endswith('.csv')]
    print(f"\nMBTI files found: {mbti_files}")

    if mbti_files:
        first_file = mbti_files[0]
        file_path = os.path.join(MBTI_PLAYLIST_PATH, first_file)

        print(f"\nExploring first file: {first_file}")
        df_mbti = pd.read_csv(file_path)

        print(f"\nShape: {df_mbti.shape}")
        print(f"\nColumns:")
        print(df_mbti.columns.tolist())

        print(f"\nFirst 3 rows:")
        print(df_mbti.head(3))

        print(f"\nData types:")
        print(df_mbti.dtypes)

        print(f"\nBasic statistics:")
        print(df_mbti.describe())

        print(f"\nNull values:")
        print(df_mbti.isnull().sum())

        print(f"\nUnique values in first column (possible ID):")
        print(df_mbti.iloc[:, 0].value_counts().head(10))

# ============================================
# 4. COMPARE DATASETS AND RECOMMENDATIONS
# ============================================

print("\n")
print("="*80)
print("COMPARISON AND RECOMMENDATIONS")
print("="*80)

print("""
DATASET SUMMARY:
================

1. AUTOENCODER DATASETS:
   - 1M songs dataset: Large dataset (167 MB) - good for pretraining autoencoder
   - 110K songs dataset: Smaller dataset (19 MB) - good for fine-tuning

   RECOMMENDATION: Use the 1M dataset for initial autoencoder training,
   then fine-tune on the 110K dataset or the playlist data.

2. CLASSIFIER DATASETS:
   - raw_playlists/: Contains individual playlists organized by MBTI type
     * Each playlist has multiple songs with their features
     * Need to aggregate features per playlist for classification

   - mbti_playlists/: Contains pre-aggregated mean features per playlist
     * Each row represents a playlist with mean audio features
     * Already labeled with MBTI type (folder/file name)

   RECOMMENDATION:
   - For classifier: Use mbti_playlists/ directly (already processed)
   - For autoencoder: Use the 1M dataset to learn rich audio feature representations
   - Consider using autoencoder to reduce dimensionality of audio features
   - Then use reduced features for MBTI classification

KEY AUDIO FEATURES TO EXPECT:
=============================
- danceability, energy, key, loudness, mode, speechiness
- acousticness, instrumentalness, liveness, valence, tempo
- duration_ms, time_signature
- (Also might have track_id, artist, etc.)

NEXT STEPS:
===========
1. Check if all datasets have the same audio feature columns
2. Standardize column names and feature ranges
3. Design autoencoder architecture
4. Train autoencoder on 1M dataset
5. Extract features from autoencoder for classifier
6. Train MBTI classifier using extracted features
""")

# Check for common columns between datasets
print("\nCOMMON FEATURES CHECK:")
print("-"*50)

if sample_1m is not None and df_playlist is not None:
    common_cols = set(sample_1m.columns).intersection(set(df_playlist.columns))
    print(f"Common columns between 1M dataset and playlist: {len(common_cols)}")
    print(f"Common numeric columns: {[c for c in common_cols if c in sample_1m.select_dtypes(include=[np.number]).columns]}")

if sample_110k is not None and df_playlist is not None:
    common_cols = set(sample_110k.columns).intersection(set(df_playlist.columns))
    print(f"\nCommon columns between 110K dataset and playlist: {len(common_cols)}")
    print(f"Common numeric columns: {[c for c in common_cols if c in sample_110k.select_dtypes(include=[np.number]).columns]}")

print("\n" + "="*80)
print("EXPLORATION COMPLETE!")
print("="*80)

Mounted at /content/drive
AUTOENCODER DATASETS EXPLORATION
1M dataset exists: True
110K dataset exists: True


EXPLORING: 1M Songs Dataset
File size: 167.68 MB

Shape (sample): (5, 20)

Columns (20 columns):
['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']

First 2 rows:
   Unnamed: 0 artist_name        track_name                track_id  \
0           0  Jason Mraz   I Won't Give Up  53QF56cjZA9RTuuMZDrSA6   
1           1  Jason Mraz  93 Million Miles  1s8tP3jP4GZcyHDsjvw218   

   popularity  year     genre  danceability  energy  key  loudness  mode  \
0          68  2012  acoustic         0.483   0.303    4   -10.058     1   
1          50  2012  acoustic         0.572   0.454    3   -10.286     1   

   speechiness  acousticness  instrumentalness  liveness  valence    tempo  \


In [9]:
# ============================================
# FINAL CLEAN DATA PREPARATION - NO PRESPLITTING
# ============================================

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Define paths
BASE_PATH = '/content/drive/MyDrive/mbti-tune'
RAW_PATH = f'{BASE_PATH}/data/raw'
CLEANED_PATH = f'{BASE_PATH}/data/cleaned'

# Create ALL necessary directories
os.makedirs(CLEANED_PATH, exist_ok=True)
os.makedirs(f'{CLEANED_PATH}/autoencoder', exist_ok=True)
os.makedirs(f'{CLEANED_PATH}/classifier', exist_ok=True)

print("="*80)
print("FINAL DATA CLEANING AND ORGANIZATION")
print("="*80)

# ============================================
# USE ONLY 10 FEATURES THAT EXIST IN ALL DATASETS
# ============================================

MASTER_FEATURES = [
    'danceability',    # How danceable
    'energy',          # Perceptual intensity
    'loudness',        # Overall loudness in dB
    'mode',            # Major=1, Minor=0
    'speechiness',     # Presence of spoken words
    'acousticness',    # Confidence of acoustic sound
    'instrumentalness', # Whether track has no vocals
    'liveness',        # Presence of live audience
    'valence',         # Musical positiveness
    'tempo'            # Beats per minute
]

print(f"\nUsing {len(MASTER_FEATURES)} features:")
for i, feat in enumerate(MASTER_FEATURES, 1):
    print(f"  {i:2d}. {feat}")

# ============================================
# HELPER FUNCTIONS
# ============================================

def clean_numeric_column(series):
    """Convert column to numeric, handling errors"""
    cleaned = pd.to_numeric(series, errors='coerce')
    if cleaned.isnull().all():
        cleaned = cleaned.fillna(0)
    else:
        cleaned = cleaned.fillna(cleaned.mean())
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    cleaned = cleaned.fillna(cleaned.mean() if not cleaned.isnull().all() else 0)
    return cleaned

def clean_and_save_dataset(df, name, folder, features, metadata_cols=None):
    """Clean and save a dataset with proper handling"""
    print(f"\nProcessing {name}...")

    # Create a copy
    df_clean = df.copy()

    # Clean feature columns
    for feature in features:
        if feature in df_clean.columns:
            df_clean[feature] = clean_numeric_column(df_clean[feature])
        else:
            print(f"   {feature} not found, adding with 0")
            df_clean[feature] = 0

    # Keep metadata columns if specified
    if metadata_cols:
        keep_cols = metadata_cols + features
        df_clean = df_clean[keep_cols]

    print(f"  Shape: {df_clean.shape}")
    print(f"  Columns: {df_clean.columns.tolist()}")
    print(f"  Data types: {df_clean.dtypes.value_counts().to_dict()}")

    # Save
    output_path = f'{CLEANED_PATH}/{folder}/{name}.csv'
    df_clean.to_csv(output_path, index=False)
    print(f"  Saved to: {output_path}")

    return df_clean

# ============================================
# 1. PROCESS AUTOENCODER DATASETS
# ============================================

print("\n" + "="*50)
print("PROCESSING AUTOENCODER DATASETS")
print("="*50)

# 1M Dataset
print("\nLoading 1M dataset...")
df_1m = pd.read_csv(f'{RAW_PATH}/spotify_data.csv')
df_1m_clean = clean_and_save_dataset(
    df_1m,
    'spotify_1m_cleaned',
    'autoencoder',
    MASTER_FEATURES
)

# 110K Dataset
print("\nLoading 110K dataset...")
df_110k = pd.read_csv(f'{RAW_PATH}/spotify_tracks.csv')
df_110k_clean = clean_and_save_dataset(
    df_110k,
    'spotify_110k_cleaned',
    'autoencoder',
    MASTER_FEATURES
)

# ============================================
# 2. PROCESS MBTI PLAYLIST DATASET (CLASSIFIER)
# ============================================

print("\n" + "="*50)
print("PROCESSING MBTI PLAYLIST DATASET (CLASSIFIER)")
print("="*50)

print("\nLoading MBTI Playlists...")

mbti_path = f'{RAW_PATH}/mbti_playlists/'
mbti_files = [f for f in os.listdir(mbti_path) if f.endswith('.csv')]

all_playlists = []

for file in mbti_files:
    mbti_type = file.replace('.csv', '')
    df = pd.read_csv(os.path.join(mbti_path, file))
    if 'mbti' not in df.columns:
        df['mbti'] = mbti_type
    all_playlists.append(df)

df_mbti = pd.concat(all_playlists, ignore_index=True)
print(f"  Combined shape: {df_mbti.shape}")

# Extract features from mean columns
df_mbti_clean = pd.DataFrame()
df_mbti_clean['mbti'] = df_mbti['mbti']
df_mbti_clean['playlist_id'] = df_mbti['playlist_id']
df_mbti_clean['track_count'] = clean_numeric_column(df_mbti['track_count'])

for feature in MASTER_FEATURES:
    mean_col = f'{feature}_mean'
    if mean_col in df_mbti.columns:
        df_mbti_clean[feature] = clean_numeric_column(df_mbti[mean_col])
    else:
        print(f"   {feature} not found, adding with 0")
        df_mbti_clean[feature] = 0

# Save
output_path = f'{CLEANED_PATH}/classifier/mbti_playlists_cleaned.csv'
df_mbti_clean.to_csv(output_path, index=False)
print(f"\n  Saved to: {output_path}")
print(f"  Shape: {df_mbti_clean.shape}")
print(f"  MBTI Distribution:")
for mbti, count in df_mbti_clean['mbti'].value_counts().items():
    print(f"    {mbti}: {count} playlists")

# ============================================
# 3. PROCESS RAW PLAYLISTS (CLASSIFIER - OPTIONAL)
# ============================================

print("\n" + "="*50)
print("PROCESSING RAW PLAYLISTS (CLASSIFIER - OPTIONAL)")
print("="*50)

column_mapping = {
    'BPM': 'tempo',
    'Energy': 'energy',
    'Dance': 'danceability',
    'Acoustic': 'acousticness',
    'Instrumental': 'instrumentalness',
    'Valence': 'valence',
    'Speech': 'speechiness',
    'Live': 'liveness',
    'Loud (Db)': 'loudness'
}

print("\nLoading Raw Playlists...")

raw_playlist_path = f'{RAW_PATH}/raw_playlists/'
personality_types = [f for f in os.listdir(raw_playlist_path)
                    if os.path.isdir(os.path.join(raw_playlist_path, f))]

all_songs = []

for personality in personality_types:
    folder_path = os.path.join(raw_playlist_path, personality)
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    for csv_file in csv_files:
        try:
            df = pd.read_csv(os.path.join(folder_path, csv_file))

            df_aligned = pd.DataFrame()
            df_aligned['mbti'] = personality
            df_aligned['playlist_id'] = csv_file.replace('.csv', '')
            df_aligned['song_index'] = df['#'].astype(str) if '#' in df.columns else range(len(df))

            # Map columns
            for raw_col, master_col in column_mapping.items():
                if raw_col in df.columns:
                    df_aligned[master_col] = clean_numeric_column(df[raw_col])
                else:
                    df_aligned[master_col] = 0

            # Add mode (not in raw playlists) - default to 1 (Major)
            df_aligned['mode'] = 1

            # Add duration if available (as milliseconds)
            if 'Duration' in df.columns:
                try:
                    parts = df['Duration'].str.split(':')
                    minutes = parts.str[0].astype(float)
                    seconds = parts.str[1].astype(float)
                    df_aligned['duration_sec'] = minutes * 60 + seconds
                except:
                    df_aligned['duration_sec'] = 0

            all_songs.append(df_aligned)
        except Exception as e:
            print(f"   Error processing {personality}/{csv_file}: {e}")
            continue

df_raw = pd.concat(all_songs, ignore_index=True)
print(f"  Combined shape: {df_raw.shape}")

# Ensure all master features exist
for feature in MASTER_FEATURES:
    if feature not in df_raw.columns:
        df_raw[feature] = 0
    df_raw[feature] = clean_numeric_column(df_raw[feature])

# Save
output_path = f'{CLEANED_PATH}/classifier/raw_playlists_cleaned.csv'
df_raw.to_csv(output_path, index=False)
print(f"\n  Saved to: {output_path}")
print(f"  Shape: {df_raw.shape}")

# ============================================
# 4. CREATE SCALED VERSIONS (FOR AUTOENCODER)
# ============================================

print("\n" + "="*50)
print("CREATING SCALED VERSIONS FOR AUTOENCODER")
print("="*50)

def extract_and_scale(df, features, scaler=None, fit=True):
    """Extract features and scale them"""
    # Extract features
    X = []
    for feature in features:
        if feature in df.columns:
            col = pd.to_numeric(df[feature], errors='coerce')
            col = col.fillna(col.mean() if not col.isnull().all() else 0)
            X.append(col.values)
        else:
            X.append(np.zeros(len(df)))

    X = np.column_stack(X)

    # Handle any remaining issues
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)

    # Scale
    if fit:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        return X_scaled, scaler
    else:
        X_scaled = scaler.transform(X)
        return X_scaled

# Extract and scale 1M dataset
print("\nScaling 1M dataset...")
X_1m_scaled, scaler = extract_and_scale(df_1m_clean, MASTER_FEATURES, fit=True)

# Save scaled 1M dataset
np.save(f'{CLEANED_PATH}/autoencoder/X_1m_scaled.npy', X_1m_scaled)
print(f"  Saved X_1m_scaled.npy: {X_1m_scaled.shape}")

# Scale 110K dataset with same scaler
print("\nScaling 110K dataset...")
X_110k_scaled = extract_and_scale(df_110k_clean, MASTER_FEATURES, scaler=scaler, fit=False)
np.save(f'{CLEANED_PATH}/autoencoder/X_110k_scaled.npy', X_110k_scaled)
print(f"  Saved X_110k_scaled.npy: {X_110k_scaled.shape}")

# Scale MBTI dataset with same scaler
print("\nScaling MBTI dataset...")
X_mbti_scaled = extract_and_scale(df_mbti_clean, MASTER_FEATURES, scaler=scaler, fit=False)
np.save(f'{CLEANED_PATH}/classifier/X_mbti_scaled.npy', X_mbti_scaled)

# Save MBTI labels separately
y_mbti = df_mbti_clean['mbti'].values
np.save(f'{CLEANED_PATH}/classifier/y_mbti.npy', y_mbti)
print(f"  Saved X_mbti_scaled.npy: {X_mbti_scaled.shape}")
print(f"  Saved y_mbti.npy: {len(y_mbti)} labels")

# Scale Raw dataset (optional)
print("\nScaling Raw dataset...")
X_raw_scaled = extract_and_scale(df_raw, MASTER_FEATURES, scaler=scaler, fit=False)
np.save(f'{CLEANED_PATH}/classifier/X_raw_scaled.npy', X_raw_scaled)
print(f"  Saved X_raw_scaled.npy: {X_raw_scaled.shape}")

# ============================================
# 5. SAVE SCALER AND METADATA
# ============================================

print("\n" + "="*50)
print("SAVING SCALER AND METADATA")
print("="*50)

# Save scaler
with open(f'{CLEANED_PATH}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("  Saved scaler.pkl")

# Save master features
with open(f'{CLEANED_PATH}/master_features.pkl', 'wb') as f:
    pickle.dump(MASTER_FEATURES, f)
print("  Saved master_features.pkl")

# Create summary CSV
summary_data = {
    'Dataset': ['1M', '110K', 'MBTI_Playlists', 'Raw_Playlists'],
    'Type': ['Autoencoder', 'Autoencoder', 'Classifier', 'Classifier'],
    'Samples': [len(df_1m_clean), len(df_110k_clean), len(df_mbti_clean), len(df_raw)],
    'Features': [len(MASTER_FEATURES)] * 4,
    'Scaled_File': [
        'X_1m_scaled.npy',
        'X_110k_scaled.npy',
        'X_mbti_scaled.npy',
        'X_raw_scaled.npy'
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f'{CLEANED_PATH}/dataset_summary.csv', index=False)
print("  Saved dataset_summary.csv")

# ============================================
# 6. FINAL SUMMARY
# ============================================

print("\n" + "="*80)
print("DATA CLEANING AND ORGANIZATION COMPLETE!")
print("="*80)

print(f"""
FINAL DATASETS:
===============

FEATURES: {len(MASTER_FEATURES)} audio features
{MASTER_FEATURES}

AUTOENCODER DATASETS (for training autoencoder):
─────────────────────────────────────────────────
1. 1M Dataset:
   - CSV: autoencoder/spotify_1m_cleaned.csv
   - Scaled: autoencoder/X_1m_scaled.npy
   - Samples: {len(df_1m_clean):,}
   - Features: {len(MASTER_FEATURES)}
   - Use: PRIMARY autoencoder training data

2. 110K Dataset:
   - CSV: autoencoder/spotify_110k_cleaned.csv
   - Scaled: autoencoder/X_110k_scaled.npy
   - Samples: {len(df_110k_clean):,}
   - Features: {len(MASTER_FEATURES)}
   - Use: Fine-tuning or additional training

CLASSIFIER DATASETS (for MBTI classification):
─────────────────────────────────────────────────
3. MBTI Playlists (PRIMARY):
   - CSV: classifier/mbti_playlists_cleaned.csv
   - Scaled: classifier/X_mbti_scaled.npy
   - Labels: classifier/y_mbti.npy
   - Samples: {len(df_mbti_clean)}
   - Features: {len(MASTER_FEATURES)}
   - MBTI Types: {len(df_mbti_clean['mbti'].unique())}
   - Use: PRIMARY classification data

4. Raw Playlists (OPTIONAL - per-song data):
   - CSV: classifier/raw_playlists_cleaned.csv
   - Scaled: classifier/X_raw_scaled.npy
   - Samples: {len(df_raw):,}
   - Features: {len(MASTER_FEATURES)}
   - Use: Per-song analysis or aggregation

METADATA:
─────────────────────────────────────────────────
- scaler.pkl: StandardScaler fitted on 1M dataset
- master_features.pkl: List of features in order
- dataset_summary.csv: Overview of all datasets

HOW TO USE:
===========

1. TRAIN AUTOENCODER:
   X_train = np.load('{CLEANED_PATH}/autoencoder/X_1m_scaled.npy')
   # Use all samples for autoencoder training
   # Or split into train/val if needed

2. FEATURE EXTRACTION:
   from sklearn.model_selection import train_test_split

   X = np.load('{CLEANED_PATH}/classifier/X_mbti_scaled.npy')
   y = np.load('{CLEANED_PATH}/classifier/y_mbti.npy')

   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

3. TRAIN CLASSIFIER:
   # First, pass X_train through autoencoder's encoder
   # Then train classifier on encoded features
   # Use the trained classifier on X_test

IMPORTANT NOTES:
===============
All datasets have EXACTLY the same {len(MASTER_FEATURES)} features
All datasets scaled with the SAME scaler (fitted on 1M dataset)
No fake data - only features that exist in ALL datasets are used
Missing values filled with column means
All columns converted to numeric
Ready for autoencoder + classifier pipeline
""")

print("="*80)
print("ALL DATASETS ARE CLEAN, ALIGNED, AND READY FOR TRAINING!")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL DATA CLEANING AND ORGANIZATION

Using 10 features:
   1. danceability
   2. energy
   3. loudness
   4. mode
   5. speechiness
   6. acousticness
   7. instrumentalness
   8. liveness
   9. valence
  10. tempo

PROCESSING AUTOENCODER DATASETS

Loading 1M dataset...

Processing spotify_1m_cleaned...
  Shape: (1159764, 20)
  Columns: ['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']
  Data types: {dtype('float64'): 9, dtype('int64'): 7, dtype('O'): 4}
  Saved to: /content/drive/MyDrive/mbti-tune/data/cleaned/autoencoder/spotify_1m_cleaned.csv

Loading 110K dataset...

Processing spotify_110k_cleaned...
  Shape: (114000, 21)
  Columns: ['Unnamed: 0', 'tr

In [11]:
# ============================================
# VERIFY CLEANED DATASETS FOR CONSISTENCY - FIXED
# ============================================

import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Define paths
BASE_PATH = '/content/drive/MyDrive/mbti-tune'
CLEANED_PATH = f'{BASE_PATH}/data/cleaned'

print("="*80)
print("VERIFYING CLEANED DATASETS FOR CONSISTENCY")
print("="*80)

# ============================================
# 1. LOAD MASTER FEATURES AND SCALER
# ============================================

print("\n" + "="*50)
print("1. LOADING METADATA")
print("="*50)

# Load master features
try:
    with open(f'{CLEANED_PATH}/master_features.pkl', 'rb') as f:
        MASTER_FEATURES = pickle.load(f)
    print(f"Loaded master features ({len(MASTER_FEATURES)} features):")
    for i, feat in enumerate(MASTER_FEATURES, 1):
        print(f"  {i:2d}. {feat}")
except Exception as e:
    print(f"Could not load master_features.pkl: {e}")
    MASTER_FEATURES = None

# Load scaler
try:
    with open(f'{CLEANED_PATH}/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    print(f"\nLoaded scaler")
    print(f"  Feature means: {scaler.mean_[:3]}...")
    print(f"  Feature stds: {scaler.scale_[:3]}...")
except Exception as e:
    print(f"Could not load scaler.pkl: {e}")
    scaler = None

# ============================================
# 2. CHECK AUTOENCODER DATASETS
# ============================================

print("\n" + "="*50)
print("2. CHECKING AUTOENCODER DATASETS")
print("="*50)

def check_dataset(name, csv_path, npy_path, features):
    """Check a dataset for consistency"""
    print(f"\nChecking: {name}")
    print("-" * 40)

    results = {}

    # Check CSV exists
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"  CSV found: {os.path.basename(csv_path)}")
        print(f"     Shape: {df.shape}")

        # Check features
        if features:
            present = [f for f in features if f in df.columns]
            missing = [f for f in features if f not in df.columns]
            print(f"     Features present: {len(present)}/{len(features)}")
            if missing:
                print(f"      Missing: {missing}")
            else:
                print(f"     All features present!")

        # Check data types for feature columns
        if features:
            print(f"     Data types (first 5 features):")
            for feat in features[:5]:
                if feat in df.columns:
                    print(f"       {feat}: {df[feat].dtype}")

        # Check for nulls in feature columns
        if features:
            nulls = df[features].isnull().sum().sum() if all(f in df.columns for f in features) else "N/A"
            print(f"     Null values in features: {nulls}")

        results['csv'] = df
    else:
        print(f"  CSV not found: {csv_path}")
        results['csv'] = None

    # Check NPY exists
    if os.path.exists(npy_path):
        X = np.load(npy_path)
        print(f"  NPY found: {os.path.basename(npy_path)}")
        print(f"     Shape: {X.shape}")
        print(f"     dtype: {X.dtype}")
        print(f"     min: {X.min():.4f}, max: {X.max():.4f}")
        print(f"     mean: {X.mean():.4f}, std: {X.std():.4f}")
        print(f"     NaN count: {np.isnan(X).sum()}")
        print(f"     Inf count: {np.isinf(X).sum()}")
        results['npy'] = X
    else:
        print(f"  NPY not found: {npy_path}")
        results['npy'] = None

    return results

# Check autoencoder datasets
autoencoder_path = f'{CLEANED_PATH}/autoencoder'

if MASTER_FEATURES:
    results_1m = check_dataset(
        "1M Dataset",
        f'{autoencoder_path}/spotify_1m_cleaned.csv',
        f'{autoencoder_path}/X_1m_scaled.npy',
        MASTER_FEATURES
    )

    results_110k = check_dataset(
        "110K Dataset",
        f'{autoencoder_path}/spotify_110k_cleaned.csv',
        f'{autoencoder_path}/X_110k_scaled.npy',
        MASTER_FEATURES
    )

# ============================================
# 3. CHECK CLASSIFIER DATASETS
# ============================================

print("\n" + "="*50)
print("3. CHECKING CLASSIFIER DATASETS")
print("="*50)

classifier_path = f'{CLEANED_PATH}/classifier'

# Check MBTI Playlists CSV
print(f"\nChecking: MBTI Playlists")
print("-" * 40)

df_mbti = None
if os.path.exists(f'{classifier_path}/mbti_playlists_cleaned.csv'):
    df_mbti = pd.read_csv(f'{classifier_path}/mbti_playlists_cleaned.csv')
    print(f"  CSV found: mbti_playlists_cleaned.csv")
    print(f"     Shape: {df_mbti.shape}")
    print(f"     Columns: {df_mbti.columns.tolist()}")

    # Check features
    if MASTER_FEATURES:
        present = [f for f in MASTER_FEATURES if f in df_mbti.columns]
        missing = [f for f in MASTER_FEATURES if f not in df_mbti.columns]
        print(f"     Features present: {len(present)}/{len(MASTER_FEATURES)}")
        if missing:
            print(f"      Missing: {missing}")
        else:
            print(f"     All features present!")

    # Check MBTI distribution
    print(f"     MBTI Distribution:")
    mbti_counts = df_mbti['mbti'].value_counts()
    for mbti, count in mbti_counts.items():
        print(f"       {mbti}: {count}")

    # Check nulls in features
    if MASTER_FEATURES:
        nulls = df_mbti[MASTER_FEATURES].isnull().sum().sum()
        print(f"     Null values in features: {nulls}")
else:
    print(f"  CSV not found: mbti_playlists_cleaned.csv")

# Check X_mbti_scaled.npy
X_mbti = None
if os.path.exists(f'{classifier_path}/X_mbti_scaled.npy'):
    X_mbti = np.load(f'{classifier_path}/X_mbti_scaled.npy')
    print(f"\n  NPY found: X_mbti_scaled.npy")
    print(f"     Shape: {X_mbti.shape}")
    print(f"     dtype: {X_mbti.dtype}")
    print(f"     min: {X_mbti.min():.4f}, max: {X_mbti.max():.4f}")
    print(f"     mean: {X_mbti.mean():.4f}, std: {X_mbti.std():.4f}")
    print(f"     NaN count: {np.isnan(X_mbti).sum()}")
    print(f"     Inf count: {np.isinf(X_mbti).sum()}")
else:
    print(f"  NPY not found: X_mbti_scaled.npy")

# Check y_mbti.npy (with allow_pickle=True for string arrays)
y_mbti = None
if os.path.exists(f'{classifier_path}/y_mbti.npy'):
    try:
        y_mbti = np.load(f'{classifier_path}/y_mbti.npy', allow_pickle=True)
        print(f"\n  NPY found: y_mbti.npy")
        print(f"     Shape: {y_mbti.shape}")
        print(f"     dtype: {y_mbti.dtype}")
        print(f"     Unique values: {np.unique(y_mbti)}")
        print(f"     Class distribution:")
        for label in np.unique(y_mbti):
            count = np.sum(y_mbti == label)
            print(f"       {label}: {count}")
    except Exception as e:
        print(f"   Could not load y_mbti.npy: {e}")
else:
    print(f"  NPY not found: y_mbti.npy")

# Check Raw Playlists (optional)
print(f"\nChecking: Raw Playlists (optional)")
print("-" * 40)

if os.path.exists(f'{classifier_path}/raw_playlists_cleaned.csv'):
    df_raw = pd.read_csv(f'{classifier_path}/raw_playlists_cleaned.csv')
    print(f"  CSV found: raw_playlists_cleaned.csv")
    print(f"     Shape: {df_raw.shape}")
    print(f"     Columns: {df_raw.columns.tolist()[:10]}...")  # Show first 10

    if MASTER_FEATURES:
        present = [f for f in MASTER_FEATURES if f in df_raw.columns]
        missing = [f for f in MASTER_FEATURES if f not in df_raw.columns]
        print(f"     Features present: {len(present)}/{len(MASTER_FEATURES)}")
        if missing:
            print(f"      Missing: {missing}")
        else:
            print(f"     All features present!")
else:
    print(f"  CSV not found: raw_playlists_cleaned.csv")

if os.path.exists(f'{classifier_path}/X_raw_scaled.npy'):
    X_raw = np.load(f'{classifier_path}/X_raw_scaled.npy')
    print(f"\n  NPY found: X_raw_scaled.npy")
    print(f"     Shape: {X_raw.shape}")
else:
    print(f"  NPY not found: X_raw_scaled.npy")

# ============================================
# 4. VERIFY ALIGNMENT BETWEEN DATASETS
# ============================================

print("\n" + "="*50)
print("4. VERIFYING ALIGNMENT BETWEEN DATASETS")
print("="*50)

if MASTER_FEATURES:
    print("\nChecking feature alignment:")

    # Get feature columns from each dataset
    datasets = {}

    if 'results_1m' in locals() and results_1m.get('csv') is not None:
        df_check = results_1m['csv']
        feature_cols = [c for c in df_check.columns if c in MASTER_FEATURES]
        datasets['1M'] = feature_cols
        print(f"  1M: {len(feature_cols)}/{len(MASTER_FEATURES)} features")

    if 'results_110k' in locals() and results_110k.get('csv') is not None:
        df_check = results_110k['csv']
        feature_cols = [c for c in df_check.columns if c in MASTER_FEATURES]
        datasets['110K'] = feature_cols
        print(f"  110K: {len(feature_cols)}/{len(MASTER_FEATURES)} features")

    if df_mbti is not None:
        feature_cols = [c for c in df_mbti.columns if c in MASTER_FEATURES]
        datasets['MBTI'] = feature_cols
        print(f"  MBTI: {len(feature_cols)}/{len(MASTER_FEATURES)} features")

    # Check if all have the same features
    if len(datasets) > 1:
        all_features = set()
        for name, features in datasets.items():
            all_features.update(features)

        if len(all_features) == len(MASTER_FEATURES):
            print("\n  ALL DATASETS HAVE THE SAME FEATURES!")
        else:
            missing_from_some = set(MASTER_FEATURES) - all_features
            if missing_from_some:
                print(f"\n   Some features missing: {missing_from_some}")

# ============================================
# 5. CHECK FEATURE DISTRIBUTIONS
# ============================================

print("\n" + "="*50)
print("5. CHECKING FEATURE DISTRIBUTIONS (SCALED DATA)")
print("="*50)

# Load scaled datasets for comparison
scaled_datasets = {}

for name, path in [
    ('1M', f'{autoencoder_path}/X_1m_scaled.npy'),
    ('110K', f'{autoencoder_path}/X_110k_scaled.npy'),
    ('MBTI', f'{classifier_path}/X_mbti_scaled.npy'),
    ('Raw', f'{classifier_path}/X_raw_scaled.npy')
]:
    if os.path.exists(path):
        X = np.load(path)
        scaled_datasets[name] = X
        print(f"\n{name} Dataset (scaled):")
        print(f"  Shape: {X.shape}")
        print(f"  Mean: {X.mean():.6f}")
        print(f"  Std: {X.std():.6f}")
        print(f"  Min: {X.min():.4f}")
        print(f"  Max: {X.max():.4f}")

# Compare means - they should be close to 0 for properly scaled data
if len(scaled_datasets) > 1:
    print("\nComparing dataset means (should be close to 0):")
    for name, X in scaled_datasets.items():
        mean_val = X.mean()
        status = "" if abs(mean_val) < 0.01 else ""
        print(f"  {status} {name}: {mean_val:.6f}")

# ============================================
# 6. SUMMARY REPORT
# ============================================

print("\n" + "="*80)
print("CONSISTENCY VERIFICATION SUMMARY")
print("="*80)

issues = []
warnings_list = []

if MASTER_FEATURES is None:
    issues.append("master_features.pkl missing")
else:
    print(f"Master features: {len(MASTER_FEATURES)} features")

if scaler is None:
    issues.append("scaler.pkl missing")
else:
    print(f"Scaler: Loaded successfully")

# Check autoencoder datasets
if os.path.exists(f'{autoencoder_path}/spotify_1m_cleaned.csv'):
    if os.path.exists(f'{autoencoder_path}/X_1m_scaled.npy'):
        print(f"1M Dataset: CSV and NPY files present")
    else:
        issues.append("1M NPY missing")
else:
    issues.append("1M CSV missing")

if os.path.exists(f'{autoencoder_path}/spotify_110k_cleaned.csv'):
    if os.path.exists(f'{autoencoder_path}/X_110k_scaled.npy'):
        print(f"110K Dataset: CSV and NPY files present")
    else:
        issues.append("110K NPY missing")
else:
    issues.append("110K CSV missing")

# Check classifier datasets
if os.path.exists(f'{classifier_path}/mbti_playlists_cleaned.csv'):
    if os.path.exists(f'{classifier_path}/X_mbti_scaled.npy'):
        if os.path.exists(f'{classifier_path}/y_mbti.npy'):
            print(f"MBTI Playlists: CSV, NPY, and labels present")
        else:
            warnings_list.append("y_mbti.npy missing (labels)")
    else:
        issues.append("X_mbti_scaled.npy missing")
else:
    issues.append("mbti_playlists_cleaned.csv missing")

# Check feature alignment
if df_mbti is not None and MASTER_FEATURES:
    feature_cols = [c for c in df_mbti.columns if c in MASTER_FEATURES]
    if len(feature_cols) == len(MASTER_FEATURES):
        print(f"All {len(MASTER_FEATURES)} features present in MBTI dataset")
    else:
        warnings_list.append(f"MBTI has {len(feature_cols)}/{len(MASTER_FEATURES)} features")

# Print issues and warnings
if issues:
    print("\nISSUES FOUND:")
    for issue in issues:
        print(f"  • {issue}")

if warnings_list:
    print("\n WARNINGS:")
    for warning in warnings_list:
        print(f"  • {warning}")

if not issues and not warnings_list:
    print("\nALL DATASETS ARE CONSISTENT AND READY FOR USE!")

print("\n" + "="*80)
print("VERIFICATION COMPLETE")
print("="*80)

# ============================================
# 7. SAMPLE DATA PREVIEW
# ============================================

print("\n" + "="*50)
print("SAMPLE DATA PREVIEW")
print("="*50)

if df_mbti is not None:
    print("\nMBTI Playlists - First 3 rows:")
    print(df_mbti.head(3).to_string())

if df_mbti is not None and MASTER_FEATURES:
    print("\nMBTI Playlists - Feature Statistics:")
    stats = df_mbti[MASTER_FEATURES].describe().round(4)
    print(stats)

# Check if labels and features match
if X_mbti is not None and y_mbti is not None:
    print(f"\nLabels match features: {X_mbti.shape[0]} = {y_mbti.shape[0]}")

# Check feature order in scaled files
if X_mbti is not None and MASTER_FEATURES:
    print(f"\nFeature order in X_mbti_scaled.npy: {MASTER_FEATURES}")

print("\n" + "="*80)
print("DATA VERIFICATION COMPLETE")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
VERIFYING CLEANED DATASETS FOR CONSISTENCY

1. LOADING METADATA
Loaded master features (10 features):
   1. danceability
   2. energy
   3. loudness
   4. mode
   5. speechiness
   6. acousticness
   7. instrumentalness
   8. liveness
   9. valence
  10. tempo

Loaded scaler
  Feature means: [ 0.53743823  0.6396699  -8.98135282]...
  Feature stds: [0.18447796 0.27050076 5.68221251]...

2. CHECKING AUTOENCODER DATASETS

Checking: 1M Dataset
----------------------------------------
  CSV found: spotify_1m_cleaned.csv
     Shape: (1159764, 20)
     Features present: 10/10
     All features present!
     Data types (first 5 features):
       danceability: float64
       energy: float64
       loudness: float64
       mode: int64
       speechiness: float64
     Null values in features: 0
  NPY found: X_1m_scaled.npy
     Shape: (1159764, 10)
     dtype: float64
 